# STCP - Queries Operacionais

Este notebook responde a perguntas concretas sobre o estado actual da frota em tempo real, com base nos dados que chegam do simulador Python via Azure IoT Central e Azure Event Hub para as tabelas Delta `bus_telemetry` e `charger_telemetry`.

## Estrutura
- Grupo 1: Autocarros
- Grupo 2: Estações de Carregamento
- Grupo 3: Serviços
- Grupo 4: Queries Combinadas

# Grupo 1 - Autocarros

## Estado Actual de Todos os Autocarros

In [0]:
# Devolve o registo mais recente de cada autocarro
# ROW_NUMBER() garante que apenas aparece uma linha por autocarro, a mais recente

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryTemperature, 1) as temperatura_celsius,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,bateria_percentagem,bateria_kwh,temperatura_celsius,latitude,longitude,ultima_actualizacao
BUS-001,headingToCharging,17.75,62.12,23.2,41.148164,-8.608651,16-10-2026 10:42:58
BUS-002,charging,45.92,160.71,26.0,41.152661,-8.579658,16-10-2026 10:42:58
BUS-003,inTransit,81.21,284.24,26.3,41.152661,-8.579658,16-10-2026 10:42:58


## Autocarro com a Bateria Mais Baixa

In [0]:
# Identifica o autocarro mais urgente em termos de necessidade de carregamento
# Inclui alerta: CRITICO abaixo de 20% (minimo seguranca), BAIXO abaixo de 50% (monitorizar), OK acima de 50%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        CASE
            WHEN batteryLevel < 20 THEN 'CRITICO - abaixo do minimo de seguranca'
            WHEN batteryLevel < 50 THEN 'BAIXO - monitorizar'
            ELSE 'OK'
        END as alerta
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryLevel ASC
    LIMIT 1
"""))

device_id,state,bateria_percentagem,bateria_kwh,alerta
BUS-001,headingToCharging,17.75,62.12,CRITICO - abaixo do minimo de seguranca


## Autocarros em Nível Crítico (abaixo de 20%)

In [0]:
# Lista todos os autocarros com bateria abaixo do nivel minimo de seguranca (20%)
# Estes autocarros nao devem sair em rota sem carregar primeiro

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    AND batteryLevel < 20
    ORDER BY batteryLevel ASC
"""))

device_id,state,bateria_percentagem,bateria_kwh,latitude,longitude,ultima_actualizacao
BUS-001,headingToCharging,17.75,62.12,41.148164,-8.608651,16-10-2026 10:42:58


## Autonomia Restante de Cada Autocarro

In [0]:
# Autonomia restante = bateria disponivel (kWh) / consumo medio (kWh/km)
# Consumo medio: 1.2 kWh/km (valor tipico urbano para CaetanoBus e.City Gold)
# Alerta baseado no SOC (%) e nao em km, para ser consistente com o limite de seguranca de 20%
# CRITICO: SOC < 20% (reserva minima de seguranca), BAIXO: SOC < 50%, OK: SOC >= 50%

spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_restante_km,
        CASE
            WHEN batteryLevel < 20 THEN 'CRITICO - abaixo do minimo de seguranca'
            WHEN batteryLevel < 50 THEN 'BAIXO - autonomia reduzida'
            ELSE 'OK'
        END as alerta
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryLevel ASC
""").show(5, truncate=False)

+---------+-----------------+-------------------+-----------+---------------------+---------------------------------------+
|device_id|state            |bateria_percentagem|bateria_kwh|autonomia_restante_km|alerta                                 |
+---------+-----------------+-------------------+-----------+---------------------+---------------------------------------+
|BUS-001  |headingToCharging|17.75              |62.12      |51.77                |CRITICO - abaixo do minimo de seguranca|
|BUS-002  |charging         |45.92              |160.71     |133.93               |BAIXO - autonomia reduzida             |
|BUS-003  |inTransit        |81.21              |284.24     |236.87               |OK                                     |
+---------+-----------------+-------------------+-----------+---------------------+---------------------------------------+



## Vista Completa de Cada Autocarro

In [0]:
# Vista consolidada com todos os campos relevantes de cada autocarro
# Autonomia calculada com consumo medio de 1.2 kWh/km
# Estado da bateria: CRITICO abaixo de 20%, BAIXO abaixo de 50%, OK acima de 50%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_km,
        round(batteryTemperature, 1) as temperatura_celsius,
        round(latitude, 6) as latitude,
        round(longitude, 6) as longitude,
        CASE
            WHEN batteryLevel < 20 THEN 'CRITICO'
            WHEN batteryLevel < 50 THEN 'BAIXO'
            ELSE 'OK'
        END as estado_bateria,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,bateria_percentagem,bateria_kwh,autonomia_km,temperatura_celsius,latitude,longitude,estado_bateria,ultima_actualizacao
BUS-001,headingToCharging,17.75,62.12,51.77,23.2,41.148164,-8.608651,CRITICO,16-10-2026 10:42:58
BUS-002,charging,45.92,160.71,133.93,26.0,41.152661,-8.579658,BAIXO,16-10-2026 10:42:58
BUS-003,inTransit,81.21,284.24,236.87,26.3,41.152661,-8.579658,OK,16-10-2026 10:42:58


## Evolução da Bateria do BUS-003 (últimas 10 leituras)

In [0]:
# Mostra a evolucao da bateria do BUS-003 ao longo do tempo
# Permite verificar que a bateria esta efectivamente a descer enquanto o autocarro esta em rota
# Autonomia calculada com consumo medio de 1.2 kWh/km

display(spark.sql("""
    SELECT
        device_id,
        round(batteryLevel, 2) as bateria_percentagem,
        round(batteryKwh, 2) as bateria_kwh,
        round(batteryKwh / 1.2, 2) as autonomia_km,
        state,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as hora
    FROM bus_telemetry_raw
    WHERE device_id = 'BUS-003'
    ORDER BY event_time DESC
    LIMIT 10
"""))

device_id,bateria_percentagem,bateria_kwh,autonomia_km,state,hora
BUS-003,81.21,284.24,236.87,inTransit,16-10-2026 10:42:58
BUS-003,79.9,279.64,233.03,charging,16-10-2026 10:42:52
BUS-003,78.58,275.04,229.2,charging,16-10-2026 10:42:45
BUS-003,77.27,270.44,225.37,charging,16-10-2026 10:42:41
BUS-003,75.95,265.84,221.53,charging,16-10-2026 10:42:35
BUS-003,74.64,261.24,217.7,charging,16-10-2026 10:42:31
BUS-003,73.33,256.64,213.87,charging,16-10-2026 10:42:25
BUS-003,72.01,252.04,210.03,charging,16-10-2026 10:42:21
BUS-003,70.7,247.44,206.2,charging,16-10-2026 10:42:14
BUS-003,69.38,242.84,202.37,charging,16-10-2026 10:42:11


# Grupo 2 - Estações de Carregamento

## Estado Actual de Todas as Estações

In [0]:
# Devolve o estado mais recente de cada estacao de carregamento
# Estados possiveis: available (livre), occupied (a carregar), fault (avaria)

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(currentPower, 1) as potencia_actual_kw,
        round(currentEfficiency, 1) as eficiencia_percentagem,
        round(chargerTemperature, 1) as temperatura_celsius,
        round(energyDelivered, 2) as energia_entregue_kwh,
        connectedBus as autocarro_ligado,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
"""))

device_id,state,potencia_actual_kw,eficiencia_percentagem,temperatura_celsius,energia_entregue_kwh,autocarro_ligado,ultima_actualizacao
CS-001,occupied,120.0,92.1,45.0,35.89,BUS-002,16-10-2026 10:42:54
CS-002,fault,0.0,0.0,28.7,0.0,null,16-10-2026 10:42:54
CS-003,occupied,120.0,91.2,35.7,23.17,BUS-003,16-10-2026 10:42:54


## Estações Disponíveis para Carregamento

In [0]:
# Lista apenas as estacoes com estado available, prontas para receber um autocarro
# Estacoes em fault ou occupied nao aparecem nesta lista

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(chargerTemperature, 1) as temperatura_celsius,
        round(currentEfficiency, 1) as eficiencia_percentagem
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    AND state = 'available'
    ORDER BY device_id
"""))

device_id,state,temperatura_celsius,eficiencia_percentagem


## Estações com Avaria

In [0]:
# Lista as estacoes com estado fault, ou seja, com avaria e fora de servico
# Estas estacoes nao podem receber autocarros ate serem reparadas

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(chargerTemperature, 1) as temperatura_celsius,
        date_format(event_time, 'dd-MM-yyyy HH:mm:ss') as ultima_actualizacao
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
    AND state = 'fault'
    ORDER BY device_id
"""))

device_id,state,temperatura_celsius,ultima_actualizacao
CS-002,fault,28.7,16-10-2026 10:42:54


# Grupo 3 - Serviços

## Capacidade Disponível na Rede de Carregamento

In [0]:
# Calcula a capacidade total da rede de carregamento e quanto ainda está disponível
# Potência máxima nominal por estação: 120 kW (CaetanoBus e.City Gold, temperatura normal)
# A potência real varia com a temperatura: 120 kW (normal), 80 kW (T>35°C), 50 kW (T<15°C)
# Potência total nominal da rede: 360 kW (3 estações x 120 kW)
# Estações em fault não contribuem para a capacidade disponível

display(spark.sql("""
    SELECT
        COUNT(*) as total_estacoes,
        SUM(CASE WHEN state = 'available' THEN 1 ELSE 0 END) as estacoes_livres,
        SUM(CASE WHEN state = 'occupied' THEN 1 ELSE 0 END) as estacoes_ocupadas,
        SUM(CASE WHEN state = 'fault' THEN 1 ELSE 0 END) as estacoes_avaria,
        round(SUM(CASE WHEN state != 'fault' THEN 120 ELSE 0 END), 1) as capacidade_nominal_kw,
        round(SUM(CASE WHEN state = 'occupied' THEN currentPower ELSE 0 END), 1) as potencia_em_uso_kw,
        round(SUM(CASE WHEN state = 'available' THEN 120 ELSE 0 END), 1) as capacidade_disponivel_kw
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    WHERE rn = 1
"""))


total_estacoes,estacoes_livres,estacoes_ocupadas,estacoes_avaria,capacidade_nominal_kw,potencia_em_uso_kw,capacidade_disponivel_kw
3,0,2,1,240,240.0,0


## Tempo Estimado para Carregar Cada Autocarro até 80%

In [0]:
# Calcula o tempo estimado para carregar cada autocarro até ao nível ideal de 80%
# Fórmula: tempo (horas) = energia necessária (kWh) / potência de carregamento (kW)
# Potência de carregamento: 80 kW (caso actual do simulador — temperatura >35°C)
# Potência nominal seria 120 kW (temperatura normal), não 150 kW
# A eficiência (92%) afecta a energia que a rede fornece, não o tempo de carregamento
# Apenas calculado para autocarros com bateria abaixo de 80%

display(spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        round(350 * 0.80, 1) as bateria_objetivo_kwh,
        round((350 * 0.80) - batteryKwh, 2) as energia_necessaria_kwh,
        CASE
            WHEN batteryLevel >= 80 THEN 'Ja carregado'
            ELSE concat(
                cast(round(((350 * 0.80) - batteryKwh) / 80, 2) as string),
                ' horas'
            )
        END as tempo_estimado_carregamento
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY batteryLevel ASC
"""))


device_id,state,bateria_actual_percentagem,bateria_actual_kwh,bateria_objetivo_kwh,energia_necessaria_kwh,tempo_estimado_carregamento
BUS-001,headingToCharging,17.75,62.12,280.0,217.88,2.72 horas
BUS-002,charging,45.92,160.71,280.0,119.29,1.49 horas
BUS-003,inTransit,81.21,284.24,280.0,-4.24,Ja carregado


## Autocarros que Precisam de Carregar antes da Próxima Rota

In [0]:
# Identifica autocarros cuja bateria actual é insuficiente para completar a rota atribuída
# Critério duplo:
#   1. Bateria abaixo do SOC mínimo de segurança (20% = 70 kWh) -> não pode sair em rota
#   2. Energia insuficiente para a rota (mesmo acima de 20%, se não chegar ao destino)
# Distâncias baseadas nos dados reais da STCP (API pública stcp.pt), seguidas pelo
# simulador Python via interpolação de GPS (ROUTE_STOPS em electric_bus_simulator.py).
# Energia necessária por rota:
#   Linha 200 (BUS-001): 8.64 km x 1.2 kWh/km = 10.37 kWh (+ 20% margem = 12.4 kWh)
#   Linha 201 (BUS-002): 8.39 km x 1.2 kWh/km = 10.07 kWh (+ 20% margem = 12.1 kWh)
#   Linha 202 (BUS-003): 9.48 km x 1.2 kWh/km = 11.38 kWh (+ 20% margem = 13.7 kWh)
# SOC mínimo operacional: 20% = 70 kWh

spark.sql("""
    SELECT
        device_id,
        state,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        CASE device_id
            WHEN 'BUS-001' THEN 12.4
            WHEN 'BUS-002' THEN 12.1
            WHEN 'BUS-003' THEN 13.7
        END as energia_necessaria_kwh,
        CASE
            WHEN batteryLevel < 20
                THEN 'CRITICO - SOC abaixo do minimo de seguranca (20%)'
            WHEN batteryKwh < CASE device_id
                WHEN 'BUS-001' THEN 12.4
                WHEN 'BUS-002' THEN 12.1
                WHEN 'BUS-003' THEN 13.7
            END
                THEN 'NECESSITA CARREGAR - energia insuficiente para a rota'
            ELSE 'OK - bateria suficiente'
        END as estado_operacional
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    ORDER BY device_id
""").show(5, truncate=False)

+---------+-----------------+--------------------------+------------------+----------------------+-------------------------------------------------+
|device_id|state            |bateria_actual_percentagem|bateria_actual_kwh|energia_necessaria_kwh|estado_operacional                               |
+---------+-----------------+--------------------------+------------------+----------------------+-------------------------------------------------+
|BUS-001  |headingToCharging|17.75                     |62.12             |12.4                  |CRITICO - SOC abaixo do minimo de seguranca (20%)|
|BUS-002  |charging         |45.92                     |160.71            |12.1                  |OK - bateria suficiente                          |
|BUS-003  |inTransit        |81.21                     |284.24            |13.7                  |OK - bateria suficiente                          |
+---------+-----------------+--------------------------+------------------+----------------------+--------

## Paragem Mais Próxima de Cada Autocarro

In [0]:
# A tabela route_stops contém 86 paragens reais (30+26+30), alinhadas com o modelo
# de GPS do simulador (ver electric_bus_simulator.py). É persistida previamente no
# catálogo do Databricks; se não existir, a query falha com 'Table or view not found'.

# Identifica a paragem actual de cada autocarro com base na distancia percorrida
# em vez do GPS, que é aproximado no simulador.
# A paragem actual é a que tem distance_meters mais próximo do distanceTraveled actual.
# Usa a tabela route_stops criada a partir dos dados GTFS reais das linhas 200, 201 e 202.

display(spark.sql("""
    WITH posicoes_actuais AS (
        SELECT device_id, latitude, longitude, batteryLevel, state, distanceTraveled
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    ),
    paragem_proxima AS (
        SELECT
            b.device_id,
            b.state as estado_autocarro,
            round(b.batteryLevel, 2) as bateria_percentagem,
            round(b.latitude, 6) as autocarro_lat,
            round(b.longitude, 6) as autocarro_lon,
            round(b.distanceTraveled, 0) as distancia_percorrida_metros,
            p.stop_sequence as numero_paragem,
            p.stop_name as paragem_mais_proxima,
            p.arrival_time as hora_prevista,
            round(p.distance_meters, 0) as distancia_paragem_metros,
            round(p.stop_lat, 6) as paragem_lat,
            round(p.stop_lon, 6) as paragem_lon,
            ROW_NUMBER() OVER (
                PARTITION BY b.device_id
                ORDER BY ABS(b.distanceTraveled - p.distance_meters) ASC
            ) as rank
        FROM posicoes_actuais b
        JOIN route_stops p ON b.device_id = p.bus_id
    )
    SELECT
        device_id as autocarro,
        estado_autocarro,
        bateria_percentagem,
        autocarro_lat,
        autocarro_lon,
        distancia_percorrida_metros,
        numero_paragem,
        paragem_mais_proxima,
        hora_prevista,
        distancia_paragem_metros,
        paragem_lat,
        paragem_lon
    FROM paragem_proxima
    WHERE rank = 1
    ORDER BY device_id
"""))

autocarro,estado_autocarro,bateria_percentagem,autocarro_lat,autocarro_lon,distancia_percorrida_metros,numero_paragem,paragem_mais_proxima,hora_prevista,distancia_paragem_metros,paragem_lat,paragem_lon
BUS-001,headingToCharging,17.75,41.148164,-8.608651,8190.0,28,Homem do Leme,06:22,8122.0,41.163454,-8.685965
BUS-002,charging,45.92,41.152661,-8.579658,10418.0,26,Viso,08:01,8390.0,41.17784,-8.64228
BUS-003,inTransit,81.21,41.152661,-8.579658,0.0,1,Av.Aliados,06:43,0.0,41.148528,-8.610667


## Quantas Paragens Faltam para Cada Autocarro Terminar a Rota

In [0]:
# A tabela route_stops contém 86 paragens reais (30+26+30), alinhadas com o modelo
# de GPS do simulador (ver electric_bus_simulator.py). É persistida previamente no
# catálogo do Databricks.

# Calcula quantas paragens faltam para cada autocarro terminar a rota, com base na
# distancia percorrida actual e nas distancias das paragens reais do GTFS.
# Total de paragens: BUS-001 tem 30 (Linha 200), BUS-002 tem 26 (Linha 201), BUS-003 tem 30 (Linha 202)

display(spark.sql("""
    WITH posicoes_actuais AS (
        SELECT device_id, batteryLevel, state, distanceTraveled
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    ),
    paragem_actual AS (
        SELECT
            b.device_id,
            b.state as estado_autocarro,
            round(b.batteryLevel, 2) as bateria_percentagem,
            round(b.distanceTraveled, 0) as distancia_percorrida_metros,
            p.stop_sequence as paragem_actual,
            p.stop_name as nome_paragem_actual,
            p.arrival_time as hora_prevista,
            ROW_NUMBER() OVER (
                PARTITION BY b.device_id
                ORDER BY ABS(b.distanceTraveled - p.distance_meters) ASC
            ) as rank
        FROM posicoes_actuais b
        JOIN route_stops p ON b.device_id = p.bus_id
    ),
    ultima_paragem AS (
        SELECT bus_id, MAX(stop_sequence) as total_paragens,
               FIRST_VALUE(stop_name) OVER (
                   PARTITION BY bus_id ORDER BY stop_sequence DESC
               ) as paragem_final,
               FIRST_VALUE(arrival_time) OVER (
                   PARTITION BY bus_id ORDER BY stop_sequence DESC
               ) as hora_chegada_prevista
        FROM route_stops
        GROUP BY bus_id, stop_name, arrival_time, stop_sequence
        QUALIFY ROW_NUMBER() OVER (PARTITION BY bus_id ORDER BY stop_sequence DESC) = 1
    )
    SELECT
        p.device_id as autocarro,
        p.estado_autocarro,
        p.bateria_percentagem,
        p.distancia_percorrida_metros,
        p.paragem_actual,
        p.nome_paragem_actual as paragem_mais_proxima,
        p.hora_prevista,
        u.total_paragens,
        (u.total_paragens - p.paragem_actual) as paragens_restantes,
        u.paragem_final as destino_final,
        u.hora_chegada_prevista as hora_chegada_prevista
    FROM paragem_actual p
    JOIN ultima_paragem u ON p.device_id = u.bus_id
    WHERE p.rank = 1
    ORDER BY p.device_id
"""))

autocarro,estado_autocarro,bateria_percentagem,distancia_percorrida_metros,paragem_actual,paragem_mais_proxima,hora_prevista,total_paragens,paragens_restantes,destino_final,hora_chegada_prevista
BUS-001,headingToCharging,17.75,8190.0,28,Homem do Leme,06:22,30,2,Castelo do Queijo,06:24
BUS-002,charging,45.92,10418.0,26,Viso,08:01,26,0,Viso,08:01
BUS-003,inTransit,81.21,0.0,1,Av.Aliados,06:43,30,29,Passeio Alegre,07:09


# Grupo 4 - Queries Combinadas

## Qual o Autocarro que Está a Ser Carregado em Cada Estação

In [0]:
# Cruza os dados das estacoes com os dados dos autocarros
# para mostrar o estado completo de cada sessao de carregamento activa
# Nota: connectedBus pode referenciar dispositivos nao presentes na tabela de autocarros
# (ex: BUS-004 hardcoded no simulador de estacoes) -> apresentado como 'N/A'

spark.sql("""
    WITH estacoes_ocupadas AS (
        SELECT device_id as estacao_id, connectedBus, currentPower, energyDelivered, chargerTemperature
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM charger_telemetry_raw
        )
        WHERE rn = 1
        AND state = 'occupied'
    ),
    autocarros AS (
        SELECT device_id, batteryLevel, batteryKwh, state as estado_autocarro
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
    )
    SELECT
        e.estacao_id as estacao,
        e.connectedBus as autocarro,
        e.currentPower as potencia_kw,
        round(e.energyDelivered, 2) as energia_entregue_kwh,
        round(e.chargerTemperature, 1) as temperatura_estacao_celsius,
        COALESCE(cast(round(a.batteryLevel, 2) as string), 'N/A') as bateria_autocarro_percentagem,
        COALESCE(cast(round(a.batteryKwh, 2) as string), 'N/A') as bateria_autocarro_kwh,
        COALESCE(a.estado_autocarro, 'N/A') as estado_autocarro
    FROM estacoes_ocupadas e
    LEFT JOIN autocarros a ON e.connectedBus = a.device_id
    ORDER BY e.estacao_id
""").show(5, truncate=False)

+-------+---------+-----------+--------------------+---------------------------+-----------------------------+---------------------+----------------+
|estacao|autocarro|potencia_kw|energia_entregue_kwh|temperatura_estacao_celsius|bateria_autocarro_percentagem|bateria_autocarro_kwh|estado_autocarro|
+-------+---------+-----------+--------------------+---------------------------+-----------------------------+---------------------+----------------+
|CS-001 |BUS-002  |120.0      |35.89               |45.0                       |45.92                        |160.71               |charging        |
|CS-003 |BUS-003  |120.0      |23.17               |35.7                       |81.21                        |284.24               |inTransit       |
+-------+---------+-----------+--------------------+---------------------------+-----------------------------+---------------------+----------------+



## Há Estações Suficientes para os Autocarros que Precisam de Carregar?

In [0]:
# Verifica se existe capacidade de carregamento disponível para os autocarros que precisam carregar
# Critério: SOC < 20% (abaixo do mínimo de segurança) OU energia insuficiente para completar a rota
#   Valores calculados com as distâncias reais da STCP (ver cell-27):
#   BUS-001: < 12.4 kWh, BUS-002: < 12.1 kWh, BUS-003: < 13.7 kWh
# Cruza o número de autocarros que precisam de carregar com o número de estações disponíveis

display(spark.sql("""
    WITH autocarros_precisam AS (
        SELECT COUNT(*) as autocarros_precisam_carregar
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM bus_telemetry_raw
        )
        WHERE rn = 1
        AND (
            batteryLevel < 20
            OR batteryKwh < CASE device_id
                WHEN 'BUS-001' THEN 12.4
                WHEN 'BUS-002' THEN 12.1
                WHEN 'BUS-003' THEN 13.7
            END
        )
    ),
    estacoes_livres AS (
        SELECT COUNT(*) as estacoes_disponiveis
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
            FROM charger_telemetry_raw
        )
        WHERE rn = 1
        AND state = 'available'
    )
    SELECT
        a.autocarros_precisam_carregar,
        e.estacoes_disponiveis,
        CASE
            WHEN e.estacoes_disponiveis >= a.autocarros_precisam_carregar THEN 'OK - capacidade suficiente'
            WHEN e.estacoes_disponiveis > 0 THEN 'ATENCAO - capacidade parcial'
            ELSE 'CRITICO - sem estacoes disponiveis'
        END as estado_capacidade,
        (e.estacoes_disponiveis - a.autocarros_precisam_carregar) as estacoes_excedentes
    FROM autocarros_precisam a, estacoes_livres e
"""))


autocarros_precisam_carregar,estacoes_disponiveis,estado_capacidade,estacoes_excedentes
1,0,CRITICO - sem estacoes disponiveis,-1


## Custo Estimado para Carregar Todos os Autocarros até 80%

In [0]:
# Calcula o custo estimado de energia para carregar todos os autocarros até ao nível ideal de 80%
# Tarifa base: 0.15 euros por kWh (definida no simulador)
# Eficiência de carregamento: 92% — afecta a energia total retirada da rede (energia_rede_necessaria_kwh)
# Tempo de carregamento: energia_necessaria / 80 kW (caso actual do simulador — temperatura >35°C)
# Apenas considera autocarros com bateria abaixo de 80%

display(spark.sql("""
    SELECT
        device_id as autocarro,
        round(batteryLevel, 2) as bateria_actual_percentagem,
        round(batteryKwh, 2) as bateria_actual_kwh,
        round(350 * 0.80, 1) as bateria_objetivo_kwh,
        round((350 * 0.80) - batteryKwh, 2) as energia_necessaria_kwh,
        round(((350 * 0.80) - batteryKwh) / 0.92, 2) as energia_rede_necessaria_kwh,
        round((((350 * 0.80) - batteryKwh) / 0.92) * 0.15, 2) as custo_estimado_euros,
        round(((350 * 0.80) - batteryKwh) / 80, 2) as tempo_estimado_horas
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    WHERE rn = 1
    AND batteryLevel < 80
    ORDER BY batteryLevel ASC
"""))


autocarro,bateria_actual_percentagem,bateria_actual_kwh,bateria_objetivo_kwh,energia_necessaria_kwh,energia_rede_necessaria_kwh,custo_estimado_euros,tempo_estimado_horas
BUS-001,17.75,62.12,280.0,217.88,236.83,35.52,2.72
BUS-002,45.92,160.71,280.0,119.29,129.66,19.45,1.49
